# 01 — Feature Extraction

**Project:** VBX — Virtual Molecular Biopsy · Manipal Hackathon 2026  
**Track:** ML · Layer 1 Data  
**Environment:** Kaggle (GPU off, internet on) or Google Colab  
**Author:** Aman Mishra  

This notebook downloads the UCSF-PDGM clinical CSV and the BraTS↔TCIA crosswalk from TCIA,  
joins them to the BraTS 2021 training set available as a Kaggle dataset, deduplicates follow-up  
scans, extracts pyradiomics features over four MRI sequences × three tumor sub-regions, applies  
the pandera data contract, creates patient-level splits, and exports the feature table.

**Leakage policy:** age and sex are the only clinical inputs.  
WHO grade, pathology diagnosis, extent of resection, treatment, survival, vital status, MGMT index,  
and all post-operative variables are NEVER used as features.

**Column naming convention:**  
- Radiomics: `{sequence}__{region}__{featureclass}__{feature}` (e.g. `t1ce__et__glcm__Contrast`)  
- Shape (sequence-independent, extracted once per region): `shape__{region}__shape__{feature}`  
- Descriptors: `desc__{name}` (e.g. `desc__total_volume_mm3`)  
- Clinical inputs: `age`, `sex`  
- Targets: `idh`, `codeletion_1p19q`, `mgmt`, `who_grade`  
- Admin: `patient_id`, `split`, `fold`  
- Flags: `missing__{region}` (e.g. `missing__ncr`)

## 0. Environment & reproducibility

In [ ]:
import sys, subprocess, importlib

def _pip(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *pkgs])

# pyradiomics requires numpy<2 on Python 3.11+ with newer SimpleITK builds.
# Pin order matters: numpy first, then SimpleITK, then pyradiomics.
_pip('numpy<2')
_pip('SimpleITK==2.3.1')
_pip('pyradiomics==3.1.0')
_pip('pandera==0.20.4', 'pyarrow', 'tqdm', 'requests')

print('Install complete')

In [ ]:
import hashlib, json, os, re, warnings
import numpy as np
import pandas as pd
import SimpleITK as sitk
import radiomics
from radiomics import featureextractor
import pandera as pa
from pathlib import Path
from tqdm.auto import tqdm
import yaml

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

print(f'numpy      {np.__version__}')
print(f'SimpleITK  {sitk.Version.VersionString()}')
print(f'pyradiomics {radiomics.__version__}')
print(f'pandas     {pd.__version__}')
print(f'pandera    {pa.__version__}')

In [ ]:
# ── Radiomics config hash ────────────────────────────────────────────────
# The config lives in ml/configs/radiomics.yaml in the repo.
# We embed it here verbatim so the notebook is self-contained on Kaggle,
# and we record its sha256 alongside library versions for reproducibility.

RADIOMICS_YAML = """
setting:
  binWidth: 25
  resampledPixelSpacing: [1, 1, 1]
  interpolator: sitkBSpline
  padDistance: 10
  normalize: true
  normalizeScale: 100
  removeOutliers: 3.0
  label: 1

featureClass:
  firstorder: []
  shape: []
  glcm: []
  glrlm: []
  glszm: []
  ngtdm: []
  gldm: []
"""

config_hash = hashlib.sha256(RADIOMICS_YAML.encode()).hexdigest()

provenance = {
    'radiomics_config_sha256': config_hash,
    'numpy_version': np.__version__,
    'SimpleITK_version': sitk.Version.VersionString(),
    'pyradiomics_version': radiomics.__version__,
    'seed': SEED,
}
print(json.dumps(provenance, indent=2))

## 1. Paths & Kaggle dataset layout

On Kaggle, add the dataset **`rsna-miccai-brain-tumor-radiogenomic-classification`** (BraTS 2021)  
and the **`ucsf-pdgm-v3`** dataset (TCIA mirror) to this notebook via *Add data*.  
Also add the TCIA crosswalk CSV as a separate dataset or upload it as a notebook file.

The code below auto-detects Kaggle vs Colab paths. On Colab, mount Drive and set  
`COLAB_BRATS_ROOT` / `COLAB_UCSF_ROOT` / `COLAB_CROSSWALK_PATH` environment variables.

In [ ]:
def _env(key, default):
    return os.environ.get(key, default)

ON_KAGGLE = Path('/kaggle/input').exists()

if ON_KAGGLE:
    # BraTS 2021 training volumes (Kaggle dataset)
    BRATS_ROOT = Path('/kaggle/input/rsna-miccai-brain-tumor-radiogenomic-classification/train')
    # UCSF-PDGM clinical CSV (Kaggle dataset or uploaded file)
    UCSF_ROOT = Path('/kaggle/input/ucsf-pdgm-v3')
    # BraTS↔TCIA crosswalk (uploaded file)
    CROSSWALK_PATH = Path('/kaggle/input/brats-tcia-crosswalk/BraTS2021_TCIA_Crosswalk.csv')
    OUT_DIR = Path('/kaggle/working')
else:
    # Colab / local: override via env vars
    BRATS_ROOT = Path(_env('COLAB_BRATS_ROOT', '/content/brats/train'))
    UCSF_ROOT = Path(_env('COLAB_UCSF_ROOT', '/content/ucsf-pdgm'))
    CROSSWALK_PATH = Path(_env('COLAB_CROSSWALK_PATH', '/content/BraTS2021_TCIA_Crosswalk.csv'))
    OUT_DIR = Path(_env('COLAB_OUT_DIR', '/content/output'))

OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'BRATS_ROOT      : {BRATS_ROOT}')
print(f'UCSF_ROOT       : {UCSF_ROOT}')
print(f'CROSSWALK_PATH  : {CROSSWALK_PATH}')
print(f'OUT_DIR         : {OUT_DIR}')
print(f'BRATS_ROOT exists: {BRATS_ROOT.exists()}')

## 2. Load UCSF-PDGM clinical CSV

In [ ]:
# Find the clinical CSV — UCSF-PDGM v3 naming
ucsf_csvs = list(UCSF_ROOT.rglob('*PDGM*.csv'))
if not ucsf_csvs:
    ucsf_csvs = list(UCSF_ROOT.rglob('*.csv'))
print('Found UCSF CSVs:', ucsf_csvs)

UCSF_CSV = ucsf_csvs[0]   # use the first (should be the clinical table)
raw_ucsf = pd.read_csv(UCSF_CSV)
print(f'Rows in raw UCSF clinical CSV: {len(raw_ucsf)}')
print(raw_ucsf.head(3))

In [ ]:
# ── Column discovery & rename to canonical names ─────────────────────────
# UCSF-PDGM column names may vary across v2/v3; we map by pattern.

def _find_col(df, patterns):
    """Return the first column whose name matches any of *patterns* (case-insensitive)."""
    for pat in patterns:
        matches = [c for c in df.columns if pat.lower() in c.lower()]
        if matches:
            return matches[0]
    return None

ID_COL    = _find_col(raw_ucsf, ['ID', 'Patient', 'Case'])
AGE_COL   = _find_col(raw_ucsf, ['Age', 'AGE'])
SEX_COL   = _find_col(raw_ucsf, ['Sex', 'Gender'])
IDH_COL   = _find_col(raw_ucsf, ['IDH'])
COD_COL   = _find_col(raw_ucsf, ['1p19q', '1p/19q', 'codeletion', 'co-deletion'])
MGMT_COL  = _find_col(raw_ucsf, ['MGMT_status', 'MGMT status', 'MGMT'])
GRADE_COL = _find_col(raw_ucsf, ['WHO_Grade', 'WHO grade', 'Grade'])

print('ID col   :', ID_COL)
print('Age col  :', AGE_COL)
print('Sex col  :', SEX_COL)
print('IDH col  :', IDH_COL)
print('1p19q col:', COD_COL)
print('MGMT col :', MGMT_COL)
print('Grade col:', GRADE_COL)

In [ ]:
# ── Select & rename ──────────────────────────────────────────────────────
keep = {ID_COL: 'patient_id_raw'}
if AGE_COL:   keep[AGE_COL]   = 'age'
if SEX_COL:   keep[SEX_COL]   = 'sex_raw'
if IDH_COL:   keep[IDH_COL]   = 'idh_raw'
if COD_COL:   keep[COD_COL]   = 'codeletion_raw'
if MGMT_COL:  keep[MGMT_COL]  = 'mgmt_raw'
if GRADE_COL: keep[GRADE_COL] = 'grade_raw'

clinical = raw_ucsf[list(keep.keys())].rename(columns=keep).copy()
print(f'Clinical subset shape: {clinical.shape}')
clinical.head(3)

## 3. Deduplicate follow-up scans

UCSF-PDGM v3 contains follow-up scans encoded as `<base_id>_FU<days>d`  
(e.g. `UCSF-PDGM-0001_FU007d`). The rule: **keep the baseline scan only** —  
i.e. the row whose ID does NOT contain `_FU`.
If a patient has only follow-up rows (rare edge case), keep the earliest (`_FU` with smallest number).

This is a leakage-prevention measure: follow-up scans are post-operative and  
may reflect treatment effects rather than the pre-operative molecular profile.

In [ ]:
clinical['is_followup'] = clinical['patient_id_raw'].str.contains('_FU', na=False)
# Extract base patient ID (strip the _FU suffix)
clinical['base_id'] = clinical['patient_id_raw'].str.replace(r'_FU\d+d$', '', regex=True)

n_before = len(clinical)
n_followup = clinical['is_followup'].sum()

# Sort so baseline rows (is_followup=False) come first, then pick first per base_id
clinical_deduped = (
    clinical
    .sort_values('is_followup')          # False=0 first
    .drop_duplicates(subset='base_id', keep='first')
    .copy()
)

n_after = len(clinical_deduped)
n_dropped = n_before - n_after

print(f'Rows before dedup : {n_before}')
print(f'Follow-up rows    : {n_followup}')
print(f'Rows after dedup  : {n_after}')
print(f'Rows dropped      : {n_dropped}  (reason: follow-up scan of same patient)')

# Use the base_id as the canonical patient_id going forward
clinical_deduped['patient_id'] = clinical_deduped['base_id']
clinical_deduped = clinical_deduped.drop(columns=['is_followup', 'base_id', 'patient_id_raw'])

## 4. Label normalisation

Encode all molecular markers as 0/1 floats. Missing labels stay `NaN`; we never impute targets.

In [ ]:
def _encode_binary(series, positive_values, negative_values):
    """Map string labels to 1.0 / 0.0 / NaN.

    positive_values : strings that map to 1
    negative_values : strings that map to 0
    Anything else (including NaN) → NaN.
    """
    s = series.astype(str).str.strip().str.lower()
    out = pd.Series(np.nan, index=series.index, dtype=float)
    out[s.isin([v.lower() for v in positive_values])] = 1.0
    out[s.isin([v.lower() for v in negative_values])] = 0.0
    return out


# IDH: mutant → 1, wildtype → 0
if 'idh_raw' in clinical_deduped:
    clinical_deduped['idh'] = _encode_binary(
        clinical_deduped['idh_raw'],
        positive_values=['mutant', 'mut', '1', 'idh mutant', 'idh-mutant'],
        negative_values=['wildtype', 'wt', '0', 'idh wildtype', 'idh-wildtype'],
    )

# 1p/19q: co-deleted → 1, non-co-deleted → 0
if 'codeletion_raw' in clinical_deduped:
    clinical_deduped['codeletion_1p19q'] = _encode_binary(
        clinical_deduped['codeletion_raw'],
        positive_values=['co-deleted', 'codel', 'deleted', '1', 'yes'],
        negative_values=['non-co-deleted', 'noncodel', 'non-deleted', '0', 'no'],
    )

# MGMT: methylated → 1, unmethylated → 0
if 'mgmt_raw' in clinical_deduped:
    clinical_deduped['mgmt'] = _encode_binary(
        clinical_deduped['mgmt_raw'],
        positive_values=['methylated', 'meth', '1', 'yes', 'positive'],
        negative_values=['unmethylated', 'unmeth', '0', 'no', 'negative'],
    )

# WHO grade: numeric
if 'grade_raw' in clinical_deduped:
    clinical_deduped['who_grade'] = pd.to_numeric(
        clinical_deduped['grade_raw'], errors='coerce'
    )

# Sex: M/Male → 1, F/Female → 0
if 'sex_raw' in clinical_deduped:
    clinical_deduped['sex'] = _encode_binary(
        clinical_deduped['sex_raw'],
        positive_values=['m', 'male', '1'],
        negative_values=['f', 'female', '0'],
    ).astype('Int64')  # nullable int

# Age: numeric
clinical_deduped['age'] = pd.to_numeric(clinical_deduped.get('age', np.nan), errors='coerce')

# Drop raw columns
raw_cols = [c for c in clinical_deduped.columns if c.endswith('_raw')]
clinical_clean = clinical_deduped.drop(columns=raw_cols)

print('Label value counts after encoding:')
for col in ['idh', 'codeletion_1p19q', 'mgmt', 'who_grade']:
    if col in clinical_clean:
        print(f'  {col}: {dict(clinical_clean[col].value_counts(dropna=False))}')

## 5. Load BraTS↔TCIA crosswalk and join

In [ ]:
crosswalk = pd.read_csv(CROSSWALK_PATH)
print(f'Crosswalk rows: {len(crosswalk)}')
print(crosswalk.head(3))
print('Columns:', list(crosswalk.columns))

In [ ]:
# Identify BraTS ID and TCIA (UCSF) ID columns in the crosswalk
brats_id_col = _find_col(crosswalk, ['BraTS', 'BRATS', 'brats'])
tcia_id_col  = _find_col(crosswalk, ['TCIA', 'UCSF', 'Patient', 'Subject'])
print(f'BraTS col: {brats_id_col}  |  TCIA col: {tcia_id_col}')

crosswalk_clean = crosswalk[[brats_id_col, tcia_id_col]].rename(columns={
    brats_id_col: 'brats_id',
    tcia_id_col : 'patient_id',
})
crosswalk_clean['patient_id'] = crosswalk_clean['patient_id'].str.strip()
crosswalk_clean['brats_id']   = crosswalk_clean['brats_id'].str.strip()

n_crosswalk = len(crosswalk_clean)

# Join: clinical_clean (patient_id = UCSF base_id) ← crosswalk → BraTS volumes
joined = crosswalk_clean.merge(clinical_clean, on='patient_id', how='inner')

n_joined = len(joined)
print(f'\nCrossWalk rows           : {n_crosswalk}')
print(f'Clinical rows (deduped)  : {len(clinical_clean)}')
print(f'Joined rows              : {n_joined}')

if n_joined < 100:
    raise RuntimeError(
        f'Join produced only {n_joined} rows — expected ~300. '
        'Check column names in crosswalk and clinical CSV before proceeding. '
        'Stop and report to Tanish per team rules.'
    )

In [ ]:
# Verify each BraTS ID exists on disk
missing_volumes = []
for bid in joined['brats_id']:
    vol_dir = BRATS_ROOT / str(bid)
    if not vol_dir.exists():
        missing_volumes.append(bid)

n_missing_vol = len(missing_volumes)
print(f'BraTS IDs with missing volume directory: {n_missing_vol}')
if n_missing_vol > 0:
    print('  First 10:', missing_volumes[:10])

# Restrict to patients whose volumes are present
available = joined[~joined['brats_id'].isin(missing_volumes)].copy()
print(f'Patients with volumes present: {len(available)}')

## 6. Patient-level train/test split and CV folds

- Stratified on IDH status (the most class-imbalanced and prognostically decisive marker).
- 80% train / 20% test.
- CV folds (5-fold) assigned on train set only.
- Patients with missing IDH labels are assigned to train (we stratify only on the known IDH values).
- No patient appears in more than one split.

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold

df = available.reset_index(drop=True).copy()

N_FOLDS = 5
TEST_FRAC = 0.20

# For stratification, use IDH where available; fill unknown with -1 as a stratum
idh_strat = df['idh'].fillna(-1).astype(int)

sss = StratifiedShuffleSplit(n_splits=1, test_size=TEST_FRAC, random_state=SEED)
train_idx, test_idx = next(sss.split(df, idh_strat))

df['split'] = 'train'
df.loc[test_idx, 'split'] = 'test'
df['fold'] = -1   # -1 = test set or unassigned

# Assign CV folds on train set
train_mask = df['split']=='train'
train_df = df[train_mask].copy()
idh_strat_train = train_df['idh'].fillna(-1).astype(int)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
for fold_i, (_, fold_val_idx) in enumerate(skf.split(train_df, idh_strat_train)):
    global_idx = train_df.iloc[fold_val_idx].index
    df.loc[global_idx, 'fold'] = fold_i

n_train = (df['split']=='train').sum()
n_test  = (df['split']=='test').sum()
print(f'Train: {n_train}  Test: {n_test}  Total: {len(df)}')
print('Fold distribution (train):')
print(df[df['split']=='train']['fold'].value_counts().sort_index())

# Verify: no patient in multiple splits
assert df['patient_id'].is_unique, 'Duplicate patient_id — split leakage!'
print('\nNo duplicate patient_ids — split integrity verified.')

In [ ]:
# Class balance per marker and split
for col in ['idh', 'codeletion_1p19q', 'mgmt', 'who_grade']:
    if col not in df:
        continue
    print(f'\n── {col} ──')
    tbl = df.groupby('split')[col].value_counts(dropna=False).unstack(fill_value=0)
    print(tbl)

## 7. Volume helpers

BraTS 2021 file layout per patient directory (Kaggle dataset):  
`{brats_id}/{brats_id}_t1.nii.gz`  
`{brats_id}/{brats_id}_t1ce.nii.gz`  
`{brats_id}/{brats_id}_t2.nii.gz`  
`{brats_id}/{brats_id}_flair.nii.gz`  
`{brats_id}/{brats_id}_seg.nii.gz`

**Segmentation labels (verified against BraTS 2021 challenge documentation):**  
- 1 = Necrotic Core (NCR)  
- 2 = Peritumoral Edema (ED)  
- 4 = Enhancing Tumor (ET)  
(label 3 is not used in BraTS 2021)

In [ ]:
SEQ_KEYS = ['t1', 't1ce', 't2', 'flair']

# BraTS 2021 segmentation label map (verified against challenge documentation)
REGION_LABELS = {
    'et' : 4,   # Enhancing Tumor
    'ncr': 1,   # Necrotic Core
    'ed' : 2,   # Peritumoral Edema
}

def _vol_path(brats_id, seq):
    """Return Path to a NIfTI volume or None if not found."""
    p = BRATS_ROOT / str(brats_id) / f'{brats_id}_{seq}.nii.gz'
    return p if p.exists() else None

def _seg_path(brats_id):
    p = BRATS_ROOT / str(brats_id) / f'{brats_id}_seg.nii.gz'
    return p if p.exists() else None

def _load_sitk(path):
    """Load a NIfTI as a SimpleITK Image; return None on failure."""
    try:
        return sitk.ReadImage(str(path))
    except Exception:
        return None

def _make_region_mask(seg_img, label):
    """Return a binary SimpleITK mask for a single segmentation label."""
    arr = sitk.GetArrayFromImage(seg_img)
    mask_arr = (arr == label).astype(np.uint8)
    mask = sitk.GetImageFromArray(mask_arr)
    mask.CopyInformation(seg_img)
    return mask

## 8. Interpretable tumor descriptors

Computed from voxel counts and the stored voxel spacing (no assumption of isotropic voxels).  
Division-by-zero is handled explicitly: result is NaN and a flag column is set.

In [ ]:
def compute_descriptors(seg_img):
    """Return dict of interpretable tumor descriptors from a segmentation image.

    All volumes are in mm³ using the actual voxel spacing.
    Division-by-zero yields NaN; a flag column records which region is missing.
    """
    arr = sitk.GetArrayFromImage(seg_img)
    spacing = seg_img.GetSpacing()   # (x, y, z) in mm
    voxel_vol_mm3 = float(spacing[0] * spacing[1] * spacing[2])

    n_et  = int((arr == REGION_LABELS['et']).sum())
    n_ncr = int((arr == REGION_LABELS['ncr']).sum())
    n_ed  = int((arr == REGION_LABELS['ed']).sum())
    n_total = n_et + n_ncr + n_ed

    desc = {}
    desc['missing__et']  = int(n_et  == 0)
    desc['missing__ncr'] = int(n_ncr == 0)
    desc['missing__ed']  = int(n_ed  == 0)

    desc['desc__total_volume_mm3'] = n_total * voxel_vol_mm3 if n_total > 0 else np.nan

    if n_total > 0:
        desc['desc__enhancing_fraction'] = n_et  / n_total
        desc['desc__necrosis_fraction']  = n_ncr / n_total
    else:
        desc['desc__enhancing_fraction'] = np.nan
        desc['desc__necrosis_fraction']  = np.nan

    n_core = n_et + n_ncr
    if n_core > 0:
        desc['desc__edema_to_core_ratio'] = n_ed / n_core
    else:
        # No core at all — ratio undefined; flag already set above
        desc['desc__edema_to_core_ratio'] = np.nan

    return desc

## 9. Pyradiomics extractor setup

Shape features are **sequence-independent** — we extract them once per region  
from the segmentation mask alone, then copy the values for all four sequences  
rather than re-running pyradiomics four times.

In [ ]:
import tempfile, pathlib

# Write YAML config to a temp file for pyradiomics
_yaml_tmp = tempfile.NamedTemporaryFile(mode='w', suffix='.yaml', delete=False)
_yaml_tmp.write(RADIOMICS_YAML)
_yaml_tmp.close()
RADIOMICS_YAML_PATH = _yaml_tmp.name

# Extractor for full feature classes (including shape)
extractor_full = featureextractor.RadiomicsFeatureExtractor(RADIOMICS_YAML_PATH)

# Shape-only extractor (for the first-sequence call; we copy the result to others)
_shape_yaml = RADIOMICS_YAML.replace(
    'firstorder: []\n  shape: []\n  glcm: []\n  glrlm: []\n  glszm: []\n  ngtdm: []\n  gldm: []',
    'shape: []'
)
# Build a minimal config dict for the shape-only extractor
_shape_config = yaml.safe_load(RADIOMICS_YAML)
_shape_config['featureClass'] = {'shape': []}
_shape_yaml_tmp = tempfile.NamedTemporaryFile(mode='w', suffix='.yaml', delete=False)
yaml.dump(_shape_config, _shape_yaml_tmp)
_shape_yaml_tmp.close()

extractor_shape = featureextractor.RadiomicsFeatureExtractor(_shape_yaml_tmp.name)
print('Extractors initialised')

In [ ]:
def _flatten_features(result_dict, prefix):
    """Filter pyradiomics result to feature keys only and apply column prefix."""
    flat = {}
    for k, v in result_dict.items():
        # pyradiomics keys: 'original_firstorder_Mean', 'original_shape_MeshVolume', etc.
        # Skip diagnostics keys (start with 'diagnostics')
        if k.startswith('diagnostics'):
            continue
        # Strip the 'original_' prefix added by pyradiomics
        k_clean = re.sub(r'^original_', '', k)
        flat[f'{prefix}__{k_clean}'] = float(v)
    return flat


def extract_patient(brats_id):
    """Extract all features for one patient. Returns a flat dict.

    Missing sequences or empty sub-regions produce NaN values plus a flag.
    Never crashes or silently fills with 0.
    """
    row = {}

    seg_path = _seg_path(brats_id)
    if seg_path is None:
        # No segmentation at all — can't extract anything
        row['missing__et']  = 1
        row['missing__ncr'] = 1
        row['missing__ed']  = 1
        return row

    seg_img = _load_sitk(seg_path)
    if seg_img is None:
        row['missing__et']  = 1
        row['missing__ncr'] = 1
        row['missing__ed']  = 1
        return row

    # ── Interpretable descriptors ────────────────────────────────────────
    row.update(compute_descriptors(seg_img))

    # ── Shape features (once per region, sequence-independent) ──────────
    for region, label in REGION_LABELS.items():
        if row.get(f'missing__{region}', 0):
            continue   # mask is empty; skip
        mask = _make_region_mask(seg_img, label)
        try:
            # pyradiomics shape features only need the mask, not the image intensity.
            # We pass the seg image as the 'image' argument (intensity unused for shape).
            res = extractor_shape.execute(seg_img, mask)
            prefix = f'shape__{region}'
            row.update(_flatten_features(res, prefix))
        except Exception as exc:
            # Record failure as NaN — do NOT silently fill
            warnings.warn(f'{brats_id} shape/{region} failed: {exc}')

    # ── Radiomics features (4 sequences × 3 regions, shape excluded) ────
    for seq in SEQ_KEYS:
        vol_path = _vol_path(brats_id, seq)
        if vol_path is None:
            # Mark all features for this sequence as missing
            # (flag columns already handled per-region above)
            continue
        vol_img = _load_sitk(vol_path)
        if vol_img is None:
            continue

        for region, label in REGION_LABELS.items():
            if row.get(f'missing__{region}', 0):
                continue
            mask = _make_region_mask(seg_img, label)
            try:
                res = extractor_full.execute(vol_img, mask)
                # Prefix: {seq}__{region}
                prefix = f'{seq}__{region}'
                # Exclude shape features — already extracted above
                for k, v in res.items():
                    if k.startswith('diagnostics') or 'shape' in k:
                        continue
                    k_clean = re.sub(r'^original_', '', k)
                    row[f'{prefix}__{k_clean}'] = float(v)
            except Exception as exc:
                warnings.warn(f'{brats_id} {seq}/{region} failed: {exc}')

    return row

## 10. Run extraction

Runs on Kaggle CPU (~45–90 min for ~300 patients).  
Progress is saved every 25 patients so a crash doesn't lose all work.

In [ ]:
import pickle

CACHE_PATH = OUT_DIR / 'radiomics_cache.pkl'

# Load partial cache if it exists (allows resuming after interruption)
if CACHE_PATH.exists():
    with open(CACHE_PATH, 'rb') as fh:
        cache = pickle.load(fh)
    print(f'Loaded cache with {len(cache)} entries')
else:
    cache = {}

rows_to_process = df[~df['brats_id'].isin(cache.keys())]
print(f'Patients remaining to process: {len(rows_to_process)}')

for i, (_, pat_row) in enumerate(tqdm(rows_to_process.iterrows(), total=len(rows_to_process))):
    bid = pat_row['brats_id']
    features = extract_patient(bid)
    cache[bid] = features

    # Save checkpoint every 25 patients
    if (i+1) % 25 == 0:
        with open(CACHE_PATH, 'wb') as fh:
            pickle.dump(cache, fh)
        print(f'  Checkpoint saved at {i+1} patients')

# Final save
with open(CACHE_PATH, 'wb') as fh:
    pickle.dump(cache, fh)
print(f'Extraction complete. Total cached: {len(cache)}')

In [ ]:
# Merge feature dicts back into the patient dataframe
feature_rows = []
for _, pat_row in df.iterrows():
    bid = pat_row['brats_id']
    feat = cache.get(bid, {})
    combined = pat_row.to_dict()
    combined.update(feat)
    feature_rows.append(combined)

features_df = pd.DataFrame(feature_rows)

# Drop brats_id from the output (not part of the contract; patient_id is the key)
features_df = features_df.drop(columns=['brats_id'], errors='ignore')

print(f'Feature table shape: {features_df.shape}')
print(f'Columns (first 20): {list(features_df.columns[:20])}')

## 11. Missing-value accounting

In [ ]:
# Count NaN per feature column
nan_counts = features_df.isna().sum()
n_feat = features_df.shape[1]
n_patients = features_df.shape[0]

print(f'Total patients     : {n_patients}')
print(f'Total columns      : {n_feat}')
print(f'\nMissing labels per marker:')
for col in ['idh', 'codeletion_1p19q', 'mgmt', 'who_grade']:
    if col in features_df:
        print(f'  {col}: {features_df[col].isna().sum()} missing')

print(f'\nMissing sequences / masks:')
for col in features_df.columns:
    if col.startswith('missing__'):
        n_flag = int(features_df[col].sum())
        print(f'  {col}: {n_flag} patients')

print(f'\nRadiomics feature columns with any NaN:')
rad_nan = nan_counts[nan_counts>0]
print(f'  {len(rad_nan)} columns have at least one NaN')

## 12. Pandera data contract validation

In [ ]:
# The schema module lives at ml/pipeline/features.py in the repo.
# Inline copy for self-contained Kaggle execution:

import pandera as pa
from pandera import Column, DataFrameSchema, Check

_ALLOWED_PREFIXES = (
    't1__', 't1ce__', 't2__', 'flair__',
    'shape__', 'desc__', 'missing__',
    'age', 'sex', 'patient_id', 'split', 'fold',
    'idh', 'codeletion_1p19q', 'mgmt', 'who_grade',
)
_KNOWN_LEAKAGE = ('resection','treatment','survival','vital','mgmt_index','post_op','diagnosis')

def _no_leakage_columns(df):
    for col in df.columns:
        col_lower = col.lower()
        for leak in _KNOWN_LEAKAGE:
            if leak in col_lower:
                return False
    return True

def _only_allowed_prefixes(df):
    for col in df.columns:
        if not any(col.startswith(p) for p in _ALLOWED_PREFIXES):
            return False
    return True

CORE_SCHEMA = DataFrameSchema(
    columns={
        'patient_id': Column(str, nullable=False),
        'split': Column(str, checks=[Check.isin(['train','test'])], nullable=False),
        'fold': Column(pa.Int64, checks=[Check.greater_than_or_equal_to(-1)], nullable=False),
        'age': Column(pa.Float64, checks=[
            Check.greater_than_or_equal_to(18), Check.less_than_or_equal_to(100)
        ], nullable=False),
        'sex': Column(pa.Int64, checks=[Check.isin([0,1])], nullable=False),
        'idh': Column(pa.Float64, checks=[Check.isin([0.0,1.0])], nullable=True),
        'codeletion_1p19q': Column(pa.Float64, checks=[Check.isin([0.0,1.0])], nullable=True),
        'mgmt': Column(pa.Float64, checks=[Check.isin([0.0,1.0])], nullable=True),
        'who_grade': Column(pa.Float64, checks=[Check.isin([2.0,3.0,4.0])], nullable=True),
        'desc__total_volume_mm3': Column(pa.Float64, checks=[Check.greater_than_or_equal_to(0)], nullable=True),
        'desc__enhancing_fraction': Column(pa.Float64, checks=[
            Check.greater_than_or_equal_to(0), Check.less_than_or_equal_to(1)
        ], nullable=True),
        'desc__necrosis_fraction': Column(pa.Float64, checks=[
            Check.greater_than_or_equal_to(0), Check.less_than_or_equal_to(1)
        ], nullable=True),
        'desc__edema_to_core_ratio': Column(pa.Float64, checks=[Check.greater_than_or_equal_to(0)], nullable=True),
    },
    checks=[
        Check(_no_leakage_columns, error='Leakage column detected'),
        Check(_only_allowed_prefixes, error='Column with disallowed prefix detected'),
        Check(lambda df: df['patient_id'].is_unique, error='Duplicate patient_id rows'),
    ],
    strict=False,
    name='VBX_FeatureTable_v1',
)

# Cast fold and sex to Int64 before validation
features_df['fold'] = features_df['fold'].astype('Int64')
features_df['sex']  = features_df['sex'].astype('Int64')

try:
    validated = CORE_SCHEMA.validate(features_df, lazy=True)
    print('✓ Pandera schema validation PASSED')
    print(f'  Rows: {len(validated)}  Columns: {len(validated.columns)}')
except pa.errors.SchemaErrors as err:
    print('✗ Schema validation FAILED:')
    print(err.failure_cases)
    raise

## 13. Hero demo cases

Shortlist 5–8 test-split patients covering a mix of IDH / 1p19q / MGMT statuses.  
We record only patient IDs here. **Volumes are not committed** pending license confirmation  
with Tanish (BraTS redistribution terms must be verified).

In [ ]:
test_df = features_df[features_df['split']=='test'].copy()

# Select patients with known labels across all markers for maximum demo variety
fully_labelled = test_df.dropna(subset=['idh','codeletion_1p19q','mgmt'])

# Try to cover all 4 combinations of IDH × MGMT, picking 1–2 per combo
hero_ids = []
for idh_val in [1.0, 0.0]:
    for mgmt_val in [1.0, 0.0]:
        subset = fully_labelled[
            (fully_labelled['idh']==idh_val) &
            (fully_labelled['mgmt']==mgmt_val)
        ]
        if len(subset) >= 2:
            hero_ids.extend(subset['patient_id'].iloc[:2].tolist())
        elif len(subset) == 1:
            hero_ids.extend(subset['patient_id'].tolist())

hero_ids = hero_ids[:8]   # cap at 8

print('Hero case patient IDs (test set):')
for pid in hero_ids:
    row = test_df[test_df['patient_id']==pid].iloc[0]
    print(f'  {pid}  IDH={row.get("idh","?"):.0f}  '
          f'1p19q={row.get("codeletion_1p19q","?"):.0f}  '
          f'MGMT={row.get("mgmt","?"):.0f}  '
          f'WHO={row.get("who_grade","?")}')

print('\nNote: volumes NOT committed. Pending BraTS license redistribution check with Tanish.')

## 14. Export feature table and compute sha256

In [ ]:
import hashlib

OUT_PARQUET = OUT_DIR / 'features.parquet'
features_df.to_parquet(OUT_PARQUET, index=False)

# sha256 of the parquet file
sha = hashlib.sha256(OUT_PARQUET.read_bytes()).hexdigest()
print(f'features.parquet written to: {OUT_PARQUET}')
print(f'sha256: {sha}')
print(f'Size  : {OUT_PARQUET.stat().st_size / 1024:.1f} KB')

provenance['feature_table_sha256'] = sha
provenance['feature_table_rows'] = len(features_df)
provenance['feature_table_cols'] = len(features_df.columns)
print('\nFull provenance record:')
print(json.dumps(provenance, indent=2))

In [ ]:
# Save provenance JSON alongside the parquet for DATA_PROVENANCE.md update
(OUT_DIR / 'provenance.json').write_text(json.dumps(provenance, indent=2))
print('provenance.json saved')

## 15. Exclusion table summary

Counts at every filtering stage, as required by docs/DATA_PROVENANCE.md.

In [ ]:
print('=' * 60)
print('EXCLUSION TABLE')
print('=' * 60)
exclusion_rows = [
    ('Rows in raw UCSF clinical CSV',           n_before,      '—'),
    ('Follow-up scans dropped (dedup rule)',     n_dropped,     'Retain baseline only; follow-up IDs contain _FU'),
    ('After dedup',                             n_after,       '—'),
    ('Rows in BraTS↔TCIA crosswalk',            n_crosswalk,   '—'),
    ('Patients after inner join',               n_joined,      'Inner join on TCIA patient ID'),
    ('Patients missing BraTS volume dir',       n_missing_vol, 'Volume not found in dataset mount'),
    ('Patients with volumes (final cohort)',     len(available),'—'),
    ('Train split',                             n_train,       'Stratified on IDH, seed=42'),
    ('Test split',                              n_test,        'Stratified on IDH, seed=42'),
]
for label, count, reason in exclusion_rows:
    print(f'  {label:<45} {count:>5}   {reason}')
print('=' * 60)

print('\nMissing labels:')
for col in ['idh', 'codeletion_1p19q', 'mgmt', 'who_grade']:
    if col in features_df:
        n_miss = features_df[col].isna().sum()
        pct = 100*n_miss/len(features_df)
        print(f'  {col:<25} {n_miss:>4} missing ({pct:.1f}%)')

## Done

Next steps (outside this notebook):
1. Download `features.parquet` and `provenance.json` from Kaggle output.
2. Copy to `artifacts/features/features.parquet`.
3. Fill `docs/DATA_PROVENANCE.md` with real counts from the exclusion table above.
4. Commit all Layer 1 files on `feat/ml-data` and merge to `main`.
5. Run CI.